Thie notebook creates simulations to contextualize the FN and FP rate for the hypothesis "cre of interest is significantly different from minP".
- negative simulations (cre_oi=minP) for quantifying the FP rate.
- positive simulations (cre_oi!=minP) for quantifying FN rate.

# Setup

In [ ]:
import sys
import os
import scMPRAforge as scm
import pandas as pd
import numpy as np
import dask.dataframe as dd

%load_ext autoreload
%autoreload 2

In [ ]:
from dask.distributed import Client, LocalCluster
cluster=LocalCluster(memory_limit='32GB')
client=Client(cluster)

In [ ]:
data_root="/nfs/roberts/project/pi_skr2/shared/tabula_data"

# Creating artificial libraries

In [ ]:
#making up the CREs
spread_gt,spread_hypothesis=scm.activity_spread(
    cell_types=list(scm.SHENDURE_BOUNDS.cells_per_cell_type.keys()),
    minimum=scm.SHENDURE_BOUNDS.min_mpra_umi,
    maximum=scm.SHENDURE_BOUNDS.max_mpra_umi,
    minp_value=scm.SHENDURE_BOUNDS.reference_activity,
    total=100,
    frac_active=0.5,
    ct_specificity=.2)

libraries=[scm.simulate_library(CREs=spread_gt["cre_id"],
                 library_model=scm.SHENDURE_BOUNDS.library_model)
                 for i in range(5)]

In [ ]:
sim=scm.de_novo_simulation(location=data_root,
                            name="pow_sim_2025-12-15",
                            libraries=libraries,
                            library_mapping="corresponding",
                            n_sims=5,
                            experiment_bounds=scm.SHENDURE_BOUNDS,
                            ground_truth=spread_gt,
                            )

In [ ]:
batch=scm.de_novo_simulation(
                        simulation_replicates=5,
                        experiment_bounds=scm.SHENDURE_BOUNDS,
                        ground_truth=spread_gt,
                        library=library)
batch.gamut(client)

In [ ]:
batch.save(f"{data_root}/simulated/shendure_pow_analysis","sim_20251119")

In [ ]:
spread_hypothesis.to_tsv(f"{data_root}/simulated/shendure_pow_analysis/spread_hypothesis_20251119.tsv")

In [ ]:
spread_gt.to_csv(f"{data_root}/simulated/shendure_pow_analysis/spread_gt_20251119.tsv",sep="\t")

In [ ]:
client.close()
cluster.close()